In [1]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_predict
from src.features import add_features
from src.model import build_model

train = add_features(pd.read_csv("../data/processed/train.csv"))  # pas le test
X_train, y_train = train.drop(columns="Price"), train["Price"]

# Chaque voiture est prédite par un modèle qui ne l'a pas vue à l'entraînement
pred = cross_val_predict(build_model(), X_train, y_train, cv=5)

errors = train.assign(pred=pred, abs_err=np.abs(pred - y_train))
errors["pct_err"] = errors["abs_err"] / errors["Price"] * 100

print("MAE :", round(errors["abs_err"].mean()))
print("Erreur absolue médiane :", round(errors["abs_err"].median()))
print("Erreur relative médiane (%) :", round(errors["pct_err"].median(), 1))

MAE : 17769
Erreur absolue médiane : 7700
Erreur relative médiane (%) : 7.9


In [2]:
errors["ecart"] = errors["pred"] - errors["Price"]  # positif = surestimation
errors["tranche_prix"] = pd.cut(
    errors["Price"], [0, 50_000, 100_000, 200_000, 400_000, 1_000_000])
errors["tranche_age"] = pd.cut(errors["car_age"], [-1, 2, 5, 10, 15, 20, 50])

def resume(col):
    return errors.groupby(col, observed=True).agg(
        n=("abs_err", "size"),
        mae=("abs_err", "mean"),
        err_pct_med=("pct_err", "median"),
        ecart_med=("ecart", "median"),
    ).round(1)

print(resume("tranche_prix"), "\n")
print(resume("tranche_age"), "\n")
top = errors["Brand"].value_counts().head(10).index
print(resume("Brand").loc[top])

                       n       mae  err_pct_med  ecart_med
tranche_prix                                              
(0, 50000]         10804    9875.5         13.6     2279.3
(50000, 100000]    16806    9321.9          8.1      191.1
(100000, 200000]   20795   13391.3          6.2    -2011.2
(200000, 400000]    8614   28924.9          6.9    -4242.6
(400000, 1000000]   1985  129700.5         11.5   -43189.9 

                 n      mae  err_pct_med  ecart_med
tranche_age                                        
(-1, 2]       3843  29567.4          6.5    -1671.9
(2, 5]        9870  23354.7          6.4     -770.6
(5, 10]      15595  18629.6          6.5     -460.0
(10, 15]     11693  15080.8          8.0     -406.0
(15, 20]      8431  13851.4          9.2        2.4
(20, 50]      9572  12605.3         13.1     -155.1 

                  n      mae  err_pct_med  ecart_med
Brand                                               
Renault        7319  12777.5          7.9     -121.3
Volkswag

In [3]:
cols = ["Brand", "Model", "car_age", "Mileage", "Condition",
        "Price", "pred", "abs_err"]
pires = errors.sort_values("abs_err", ascending=False).head(20)
print(pires[cols].round(0).to_string(index=False))

        Brand             Model  car_age           Mileage Condition     Price     pred  abs_err
        Dacia             Logan      6.0   10 000 - 14 999 Excellent  980000.0  23161.0 956839.0
Mercedes-Benz               250     32.0     5 000 - 9 999       New  999999.0  56516.0 943483.0
         Fiat               500      7.0   10 000 - 14 999 Very Good 1000000.0  91574.0 908426.0
      Hyundai            Tucson     16.0 160 000 - 169 999 Excellent  980000.0  76397.0 903603.0
      Peugeot              3008     12.0   20 000 - 24 999 Very Good 1000000.0 102375.0 897625.0
Mercedes-Benz              310D     33.0   30 000 - 34 999       NaN 1000000.0 104288.0 895712.0
      Hyundai          Santa Fe     12.0 180 000 - 189 999 Very Good 1000000.0 112621.0 887379.0
          Kia           Picanto      5.0   55 000 - 59 999 Excellent  980000.0 105246.0 874754.0
      Citroen                C4     11.0 200 000 - 249 999 Excellent  970000.0  97397.0 872603.0
   Volkswagen            Passa